# Addis Ababa PM2.5 — Next-Day Forecast (Regression)

**Problem:** predict **tomorrow's** daily PM2.5 (µg/m³) from today's air-quality values and the recent past.
**Metric:** RMSE (also MAE, R²). **Success:** beat the persistence baseline ("tomorrow = today") by ≥ 10% RMSE.

## 1. Setup & load

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

Clean_Path = Path("../data/processed")/"air_quality_clean.csv"
Fig_Dir = Path("../reports/figures")
Fig_Dir.mkdir(parents=True, exist_ok=True)

df_clean = pd.read_csv(Clean_Path, parse_dates=["date"])

df_clean.shape

(1294, 13)

## 2. Target & features
- **Target:** tomorrow's PM2.5 (`shift(-1)`).
- **Lags:** PM2.5 yesterday and 2 days ago; **rolling:** 7-day mean (includes today, which is known at prediction time).
- `us_aqi` and `european_aqi` are excluded because they are derived from PM2.5 (target leakage).

In [2]:
df_clean["target"] = df_clean["pm2_5"].shift(-1)
df_clean["target"].tail()

1289    15.566667
1290    17.920833
1291    17.887500
1292    19.312500
1293          NaN
Name: target, dtype: float64

In [3]:
df_clean["pm25_lag1"] = df_clean["pm2_5"].shift(1)
df_clean["pm25_lag2"] = df_clean["pm2_5"].shift(2)

print(df_clean["pm25_lag1"].head())
df_clean["pm25_lag2"].head()

0          NaN
1    12.250000
2     9.787500
3    10.287500
4    13.229167
Name: pm25_lag1, dtype: float64


0        NaN
1        NaN
2    12.2500
3     9.7875
4    10.2875
Name: pm25_lag2, dtype: float64

In [4]:
df_clean["pm25_roll7"] = df_clean["pm2_5"].rolling(7).mean()
df_clean["pm25_roll7"]

0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
          ...    
1289    15.752381
1290    15.072024
1291    15.450595
1292    16.004167
1293    16.780357
Name: pm25_roll7, Length: 1294, dtype: float64

In [5]:
df_clean = df_clean.dropna().reset_index(drop=True)
df_clean.shape
df_clean.isna().sum()



date                     0
pm10                     0
pm2_5                    0
carbon_monoxide          0
nitrogen_dioxide         0
sulphur_dioxide          0
ozone                    0
aerosol_optical_depth    0
dust                     0
uv_index                 0
us_aqi                   0
european_aqi             0
month                    0
target                   0
pm25_lag1                0
pm25_lag2                0
pm25_roll7               0
dtype: int64

In [6]:
Features = ['pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide',
       'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index','month',
       'pm25_lag1', 'pm25_lag2','pm25_roll7']


X = df_clean[Features]
y = df_clean["target"]


In [7]:
X.shape

(1287, 13)

In [8]:
print("target" in X)
print("us_aqi" in X)

False
False


In [9]:
print(df_clean["date"].iloc[[0, -1]])
print((df_clean["target"].iloc[:-1].values == df_clean["pm2_5"].iloc[1:].values).all())

0      2022-08-11
1286   2026-02-17
Name: date, dtype: datetime64[us]
True


## 3. Train/test split (by time)
The last 20% of days is the test set (no shuffling), so the model is always tested on the future.
**Validation:** instead of one fixed validation set, `TimeSeriesSplit(5)` on the training part gives 5 rolling validation windows (always train on the past, validate on the next block). All model choices use these windows; the test set is used once at the end.

In [10]:
split = int(len(X)*0.8)

In [11]:
# train_test_split with shuffle parameter off can be used 
X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

## 4. Baselines

In [12]:
from sklearn.metrics import root_mean_squared_error, mean_squared_error, r2_score, mean_absolute_error

# persistence baseline: tomorrow's PM2.5 = today's PM2.5 (no training needed)
rmse = root_mean_squared_error(y_test, X_test["pm2_5"])
print(rmse)

mae = mean_absolute_error(y_test, X_test["pm2_5"])
print(mae)

r2 = r2_score(y_test, X_test["pm2_5"])
print(r2)

4.188353911675653
3.2081072351421187
0.47622480929087463


In [13]:
from sklearn.dummy import DummyRegressor

# mean baseline: always predict the training-set average
model = DummyRegressor(strategy='mean')

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

mae = mean_absolute_error(y_test, y_pred)
print(mae)

r2 = r2_score(y_test, y_pred)
print(r2)

6.605309627450739
4.962065758381609
-0.3027003807568447


**Baseline results (test set, from 2025-06-05)**

| Baseline | MSE | RMSE | MAE | R² |
|---|---|---|---|---|
| Persistence | 17.54 | 4.19 | 3.21 | 0.48 |
| Mean | 43.63 | 6.61 | 4.96 | −0.30 |

The mean baseline's negative R² reflects the shift between the training and test periods. **Target for the models: RMSE ≤ 3.77.**

## 5. Univariate & bivariate models
Plain linear regression with only 1 or 2 inputs, to see how much each extra feature adds:
- **Univariate:** today's PM2.5 only (ŷ = w·x + b).
- **Bivariate:** today's PM2.5 + the 7-day mean (short-term value + trend).

No scaling or regularization is needed with so few features. These models have no tuning, so testing them now is fair, like the baselines.

In [18]:
# returns MSE, RMSE, MAE and R² for one set of predictions
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MSE": mse,
            "RMSE": float(np.sqrt(mse)),
            "MAE": mean_absolute_error(y_true, y_pred),
            "R2": r2_score(y_true, y_pred)}

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.linear_model import LinearRegression

tscv = TimeSeriesSplit(n_splits=5)

feature_sets = {
    "Univariate": ["pm2_5"],
    "Bivariate": ["pm2_5", "pm25_roll7"],
}

simple_models = {}
for name, cols in feature_sets.items():
    model = LinearRegression()
    cv_res = cross_validate(model, X_train[cols], y_train, cv=tscv, scoring="neg_root_mean_squared_error")
    model.fit(X_train[cols], y_train)
    simple_models[name] = model

    m = evaluate(y_test, model.predict(X_test[cols]))
    print(f"{name}: CV RMSE {-cv_res['test_score'].mean():.3f} | "
          f"test MSE {m['MSE']:.2f}, RMSE {m['RMSE']:.3f}, MAE {m['MAE']:.3f}, R² {m['R2']:.3f}")
    print("   coef:", dict(zip(cols, model.coef_.round(3))), "| intercept:", round(model.intercept_, 3))

In [ ]:
# univariate fit vs persistence: the learned line is flatter than ŷ = x
uni = simple_models["Univariate"]
x_line = np.linspace(X_train["pm2_5"].min(), X_train["pm2_5"].max(), 100)

plt.figure(figsize=(6.5, 5))
plt.scatter(X_train["pm2_5"], y_train, s=8, alpha=0.4, label="training days")
plt.plot(x_line, uni.predict(pd.DataFrame({"pm2_5": x_line})), color="red",
         label=f"univariate fit: ŷ = {uni.coef_[0]:.2f}·x + {uni.intercept_:.2f}")
plt.plot(x_line, x_line, "k--", linewidth=0.8, label="persistence: ŷ = x")
plt.xlabel("PM2.5 today (µg/m³)")
plt.ylabel("PM2.5 tomorrow (µg/m³)")
plt.title("Univariate model")
plt.legend()
plt.savefig(Fig_Dir / "univariate_fit.png")
plt.show()

**Results**

| Model | Features | CV RMSE | Test MSE | Test RMSE | Test MAE | Test R² |
|---|---|---|---|---|---|---|
| Persistence | copy of today | – | 17.54 | 4.19 | 3.21 | 0.48 |
| Univariate | 1 | 3.20 | 15.73 | 3.97 | 3.00 | 0.53 |
| Bivariate | 2 | 3.20 | 15.47 | 3.93 | 3.00 | 0.54 |

- **Univariate:** ŷ = 0.81·x + 3.24. A slope below 1 pulls tomorrow back toward the average (regression to the mean), which is why it already beats plain persistence (slope 1).
- **Bivariate:** the 7-day mean gets a small weight (0.06) and improves the result only slightly.

## 6. Model comparison (time-series CV)
Candidates chosen to cover simple → complex:

| Model | Complexity | Interpretability | Compute cost |
|---|---|---|---|
| Ridge (linear) | Low | High: one weight per feature | Very low |
| Random Forest | Medium–high | Medium: feature importances only | Medium |
| HistGradientBoosting | High | Low–medium | Low–medium |

Neural networks were not used: ~1,000 training rows is too little data for them to beat these models.

In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

model_ridge = make_pipeline(StandardScaler(), Ridge())
model_RFS = RandomForestRegressor(random_state=42)
model_HGBR = HistGradientBoostingRegressor(random_state=42)


cv_results_ridge = cross_validate(model_ridge, X_train, y_train, cv=tscv, scoring="neg_root_mean_squared_error")
cv_results_RFS = cross_validate(model_RFS, X_train, y_train, cv=tscv, scoring="neg_root_mean_squared_error")
cv_results_HGBR = cross_validate(model_HGBR, X_train, y_train, cv=tscv, scoring="neg_root_mean_squared_error")


print(-cv_results_ridge["test_score"].mean(), cv_results_ridge["test_score"].std())
print(-cv_results_RFS["test_score"].mean(), cv_results_RFS["test_score"].std())
print(-cv_results_HGBR["test_score"].mean(), cv_results_HGBR["test_score"].std())

3.447825532798082 0.7469233443592793
3.4780825374048505 0.3992923126248981
3.61143899164191 0.4044623267009185


**CV results (train only, mean ± std RMSE)**

| Model | RMSE |
|---|---|
| Ridge (scaled) | 3.45 ± 0.75 |
| RandomForest | 3.48 ± 0.40 |
| HistGradientBoosting | 3.61 ± 0.40 |

Ridge and RandomForest are tied (the difference is far smaller than the std), so we pick the simpler model, **Ridge**, to tune.

## 7. Tuning Ridge

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {"ridge__alpha": [10, 30, 100, 300, 1000, 3000]}

search = GridSearchCV(model_ridge, param_grid, cv=tscv, scoring="neg_root_mean_squared_error",
                      return_train_score=True)

search.fit(X_train, y_train)

print(search.best_params_)
print(-search.best_score_)

{'ridge__alpha': 30}
3.281326847575979


Best α = 30 (inside the grid), CV RMSE **3.28** vs 3.45 with the default α.

### Underfitting / overfitting check
Compare training error with validation error: a large gap means **overfitting** (the model memorises the training data); both errors high means **underfitting**.

In [ ]:
# train vs validation RMSE on the same time-series folds
fit_models = {
    "Univariate (linear)": (LinearRegression(), ["pm2_5"]),
    "Ridge (α=30)": (search.best_estimator_, Features),
    "Random Forest": (model_RFS, Features),
    "HistGradientBoosting": (model_HGBR, Features),
}
rows = {}
for name, (m, cols) in fit_models.items():
    res = cross_validate(m, X_train[cols], y_train, cv=tscv,
                         scoring="neg_root_mean_squared_error", return_train_score=True)
    rows[name] = {"Train RMSE": -res["train_score"].mean(), "Validation RMSE": -res["test_score"].mean()}
fit_table = pd.DataFrame(rows).T
fit_table["Gap"] = fit_table["Validation RMSE"] - fit_table["Train RMSE"]
print(fit_table.round(2))

# left: train vs validation per model; right: Ridge validation curve over alpha
cvr = pd.DataFrame(search.cv_results_)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fit_table[["Train RMSE", "Validation RMSE"]].plot(kind="bar", ax=axes[0], rot=15, color=["tab:green", "tab:orange"])
axes[0].set_ylabel("RMSE (µg/m³)")
axes[0].set_title("Train vs validation error")
axes[1].semilogx(cvr["param_ridge__alpha"].astype(float), -cvr["mean_train_score"], "o-", label="train")
axes[1].semilogx(cvr["param_ridge__alpha"].astype(float), -cvr["mean_test_score"], "o-", label="validation")
axes[1].set_xlabel("alpha (log scale)")
axes[1].set_ylabel("RMSE (µg/m³)")
axes[1].set_title("Ridge validation curve")
axes[1].legend()
plt.tight_layout()
plt.savefig(Fig_Dir / "fit_diagnosis.png")
plt.show()

**Findings**

| Model | Train RMSE | Validation RMSE | Gap | Diagnosis |
|---|---|---|---|---|
| Univariate | 3.25 | 3.20 | −0.05 | Good fit, no overfitting |
| **Ridge (α=30)** | 3.15 | 3.28 | 0.13 | **Good fit**, small gap |
| Random Forest | 1.28 | 3.48 | 2.20 | **Overfitting** |
| HistGradientBoosting | 1.17 | 3.61 | 2.45 | **Overfitting** |

- The tree models memorise the training days (train error ~1.2) but do worse on new days, so they overfit.
- The Ridge validation curve shows both failure modes: small α → train error drops below validation (starting to overfit); large α → both errors rise together (**underfitting**). α = 30 sits at the sweet spot.

## 8. Final test (used once)

In [ ]:
# best_estimator_ is already refit on the full training set
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print({k: round(v, 3) for k, v in evaluate(y_test, y_pred).items()})

(3.926518413342828, 3.0574413722777773, 0.5396655721423161)


**Test results (from 2025-06-05)**

| Model | MSE | RMSE | MAE | R² |
|---|---|---|---|---|
| Persistence | 17.54 | 4.19 | 3.21 | 0.48 |
| Ridge (α=30) | **15.42** | **3.93** | **3.06** | **0.54** |

Ridge beats persistence on all metrics, but the RMSE gain (6%) misses the 10% target (≤ 3.77). Test RMSE is higher than CV (3.28) because the test period differs from training (distribution shift).

**Note:** Ridge performs the same as the 2-feature model (test RMSE 3.93 vs 3.93, CV 3.28 vs 3.20). The extra features add almost no information, so a simpler model would work equally well. We keep Ridge as the final model because it was selected and tuned before the test.

## 9. Baselines vs models
Left: test RMSE from the simplest approach to the full model, against the target. Right: how the Ridge forecast follows the real PM2.5 over the test period.

In [ ]:
test_rmse = {
    "Mean": root_mean_squared_error(y_test, DummyRegressor().fit(X_train, y_train).predict(X_test)),
    "Persistence": root_mean_squared_error(y_test, X_test["pm2_5"]),
}
for name, cols in feature_sets.items():
    test_rmse[name] = root_mean_squared_error(y_test, simple_models[name].predict(X_test[cols]))
test_rmse["Ridge (13 features)"] = root_mean_squared_error(y_test, y_pred)
test_dates = df_clean["date"].iloc[split:]

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={"width_ratios": [1.5, 2.5]})

bars = axes[0].bar(test_rmse.keys(), test_rmse.values(), color=["grey", "grey", "tab:cyan", "tab:cyan", "tab:blue"])
axes[0].bar_label(bars, fmt="%.2f")
axes[0].axhline(3.77, color="red", linestyle="--", label="target (3.77)")
axes[0].set_ylabel("Test RMSE (µg/m³)")
axes[0].set_title("Test RMSE: lower is better")
axes[0].tick_params(axis="x", rotation=20)
axes[0].legend()

axes[1].plot(test_dates, y_test.values, label="actual", linewidth=1)
axes[1].plot(test_dates, y_pred, label="Ridge forecast", linewidth=1)
axes[1].set_ylabel("PM2.5 tomorrow (µg/m³)")
axes[1].set_title("Actual vs predicted (test period)")
axes[1].legend()

plt.tight_layout()
plt.savefig(Fig_Dir / "baseline_vs_model.png")
plt.show()

**Findings**
- Every step beats the baselines, but the gain from 1 → 2 → 13 features is tiny (test RMSE 3.97 → 3.93 → 3.93). Almost all of the predictive signal is in **today's PM2.5**; the other pollutants add little for next-day forecasting.
- The forecast follows the seasonal level and most day-to-day movement, but it often reacts about a day late (it leans heavily on today's value) and under-predicts the sharpest peaks. Weather data such as rain and wind could help.

## 10. Save the model
The saved pipeline includes the scaler, so the app can pass raw feature values straight in.

In [ ]:
import joblib

Model_Dir = Path("../models")
Model_Dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, Model_Dir / "ridge_pm25.joblib")

# reload to confirm it predicts the same
loaded = joblib.load(Model_Dir / "ridge_pm25.joblib")
print((loaded.predict(X_test) == y_pred).all())

## 11. Monitoring & maintenance plan
The model is deployed as a Streamlit app (one prediction per request, response in well under a second). In production we would:
1. **Monitor accuracy:** each day, compare yesterday's forecast with the real PM2.5 and track a rolling 30-day RMSE; alert if it rises above ~4.2 (the persistence baseline).
2. **Watch for data drift:** compare incoming feature distributions with the training data. This dataset already showed drift (CO fell from ~870 in 2022 to ~360 in 2024).
3. **Retrain:** monthly, or when the RMSE alert fires, on all data so far with the same time-based validation.
4. **Collect feedback:** log inputs, predictions and actual values to find where the model fails (e.g. sharp peaks) and add features such as weather.